# Partie 2 : étude principale pass@k, session 1 (test confirmatoire)

Plan gelé : `docs/partie2/plan_analyse.md` (v1.0). Session 1 : 500 questions de ponts HotpotQA (graine 42), 32 tirages
par question, au plus 4 recherches, température 0,7, pour les deux paires Instruct (le modèle de départ est commun).
Le run du modèle de départ est découpé en deux moitiés pour équilibrer les deux T4. Reprise automatique : relancer le
notebook complète les questions manquantes.


In [ ]:
BRANCH = "partie2-passk"
TYPE, N_QUESTIONS, N_SAMPLES, T, TEMP = "bridge", 500, 32, 4, 0.7
MAX_TOKENS, MAX_MODEL_LEN = 1024, 8192   # figés au pilote v2 ; arrêt à </search> et </answer>
QWEN_IT = "Qwen/Qwen2.5-3B-Instruct"
R = "PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-3b-"
# (modèle, options) ; une file par GPU, environ 7 h chacune d'après le débit du pilote.
FILES = [
    [(R + "it-em-ppo-v0.3", []), (QWEN_IT, ["--shard", "0/2"])],
    [(R + "it-em-grpo-v0.3", []), (QWEN_IT, ["--shard", "1/2"])],
]


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
from pathlib import Path
REPO = "vqlentinn/memoire-rl-search"
CLONE = Path("/tmp/memoire-rl-search")
if not CLONE.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(CLONE)], check=True)
print(subprocess.run(["git", "-C", str(CLONE), "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q vllm==0.33.0 pandas pyarrow huggingface_hub 2>&1 | tail -3
!pip freeze > /kaggle/working/pip_freeze.txt
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| gpus', torch.cuda.device_count())"
import sys
sys.path.insert(0, str(CLONE))
from data.hotpot_distractor import telecharger
PARQUET = telecharger("/tmp/hotpot")
print(PARQUET)

In [ ]:
import os, threading, time
WORK = Path("/kaggle/working")
(WORK / "logs").mkdir(exist_ok=True)
status, verrou = [], threading.Lock()

def file_de_runs(gpu, runs):
    env = dict(os.environ, VLLM_WORKER_MULTIPROC_METHOD="spawn", CUDA_VISIBLE_DEVICES=str(gpu))
    for model, opts in runs:
        nom = model.split("/")[-1] + ("_brut" if "--no-chat-template" in opts else "") + \
            ("_shard" + opts[opts.index("--shard") + 1].replace("/", "of") if "--shard" in opts else "")
        cmd = ["python", "-u", "scripts/run_passk.py", "--model", model, "--type", TYPE,
               "--n-questions", str(N_QUESTIONS), "--n-samples", str(N_SAMPLES), "--max-turns", str(T),
               "--temperature", str(TEMP), "--data", PARQUET,
               "--max-tokens", str(MAX_TOKENS), "--max-model-len", str(MAX_MODEL_LEN),
               "--outdir", str(WORK / "results" / "partie2")] + opts
        t0 = time.time()
        with open(WORK / "logs" / f"{nom}.log", "a") as f:
            code = subprocess.run(cmd, cwd=CLONE, env=env, stdout=f, stderr=subprocess.STDOUT).returncode
        with verrou:
            status.append((nom, gpu, code, round(time.time() - t0)))
            print(f"GPU {gpu} : {nom} code {code}, {time.time() - t0:.0f} s", flush=True)

fils = [threading.Thread(target=file_de_runs, args=(g, runs)) for g, runs in enumerate(FILES)]
for f in fils: f.start()
for f in fils: f.join()

In [ ]:
import json
from collections import Counter
resume = {}
for fichier in sorted((WORK / "results" / "partie2").glob("*.jsonl")):
    L = [json.loads(l) for l in fichier.read_text().split("\n") if l.strip()]
    if not L:
        continue
    st = Counter(l["status"] for l in L)
    par_q = Counter(l["id"] for l in L)
    resume[fichier.stem] = {
        "trajectoires": len(L), "questions_completes": sum(v >= N_SAMPLES for v in par_q.values()),
        "em": round(sum(l["em"] for l in L) / len(L), 3), "statuts": dict(st),
        "tronquees": round(sum(l["truncated"] for l in L) / len(L), 3),
        "recherches": round(sum(l["n_search"] for l in L) / len(L), 2),
        "jetons": round(sum(l["gen_tokens"] for l in L) / len(L)),
    }
(WORK / "resume_principal.json").write_text(json.dumps({"status": status, "resume": resume}, indent=1, ensure_ascii=False))
print(json.dumps({"status": status, "resume": resume}, indent=1, ensure_ascii=False))
echecs = [s for s in status if s[2] != 0]
if echecs:
    raise RuntimeError(f"Runs en échec : {echecs}")
